# Z' → tt̄ with GRAEP

End-to-end notebook for the Z' → tt̄ analysis using GRAEP. Each section adds another step to the pipeline.

Right now the notebook covers:

1. Building the fileset.
2. Exporting `nanoaods.json` and the per-process JSONs to disk.

In [1]:
import sys
from pathlib import Path

_REPO_ROOT = "/Users/mohamedaly/work/repos/GRAEP"
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

from graep.logging import setup_logging  # noqa: E402

setup_logging(3)

## 1. Build the fileset

Pick a resolver, hand it the query callables defined in
`examples/example_opendata_cms/queries.py`. `build_fileset` returns a Python dict; nothing is opened on disk yet.

In [2]:
from examples.example_opendata_cms.config import config  # noqa: E402
from graep.inputs import build_fileset  # noqa: E402

# `max_files_per_sample=2` keeps the export fast while you iterate;
# drop the kwarg (or set it to None) for the full set.
fileset = build_fileset(config.inputs, max_files_per_sample=2)

print(f"resolved {len(fileset)} processes\n")
for key, entry in fileset.items():
    md = entry["metadata"]
    print(
        f"  {key:30s} files={len(entry['files']):>3}  "
        f"xsec={md['xsec']!s:>8}  is_data={md['is_data']}"
    )


[INFO:graep.inputs.queries:_load_or_compute:L.170] fileset cache miss → wrote /tmp/graep/.cache/2f692357b84611203e8452d7a445e86c.json


resolved 6 processes

  signal__nominal                files=  2  xsec= 0.01895  is_data=False
  ttbar_semilep__nominal         files=  2  xsec=  364.31  is_data=False
  ttbar_had__nominal             files=  2  xsec=  380.11  is_data=False
  ttbar_lep__nominal             files=  2  xsec=   87.33  is_data=False
  wjets__nominal                 files=  2  xsec= 61526.7  is_data=False
  data__nominal                  files=  2  xsec=     1.0  is_data=True


## 2. Export to disk

`export_fileset` opens every file once with uproot, counts entries, sums `genWeight` for MC (skips it for processes flagged `is_data=True`), and writes the JSONs the rest of the pipeline reads.

With `max_files_per_sample=2` above, this takes about half a minute. The full set takes around 15 minutes.

In [3]:
import json  # noqa: E402
import time  # noqa: E402

from graep.inputs import export_fileset  # noqa: E402

EXPORT_DIR = Path(_REPO_ROOT) / "examples" / "example_opendata_cms" / "datasets"

t0 = time.perf_counter()
export_fileset(fileset, EXPORT_DIR, weights_branch="genWeight")
dt = time.perf_counter() - t0

nanoaods_path = EXPORT_DIR / "nanoaods.json"
per_proc_dir = EXPORT_DIR / "nanoaods_jsons_per_process"
per_proc = sorted(per_proc_dir.glob("*.json")) if per_proc_dir.exists() else []

print(f"export ran in {dt:.1f}s")
print(f"combined: {nanoaods_path}")
print(f"per-process ({len(per_proc)}):")
for p in per_proc:
    print(f"    {p.name}")

print("\ntotals from nanoaods.json:")
with nanoaods_path.open() as f:
    on_disk = json.load(f)
for proc, vars_ in on_disk.items():
    for variation, payload in vars_.items():
        print(
            f"  {proc}__{variation:8s} files={len(payload['files']):>3}  "
            f"nevts_total={payload['nevts_total']:>10}  "
            f"nevts_wt_total={payload['nevts_wt_total']:>14.2f}"
        )

[INFO:graep.inputs.export:export_fileset:L.86] wrote combined fileset JSON: /Users/mohamedaly/work/repos/GRAEP/examples/example_opendata_cms/datasets/nanoaods.json


[INFO:graep.inputs.export:export_fileset:L.98] wrote per-(process, variation) JSONs under: /Users/mohamedaly/work/repos/GRAEP/examples/example_opendata_cms/datasets/nanoaods_jsons_per_process


export ran in 16.4s
combined: /Users/mohamedaly/work/repos/GRAEP/examples/example_opendata_cms/datasets/nanoaods.json
per-process (6):
    nanoaods_data_nominal.json
    nanoaods_signal_nominal.json
    nanoaods_ttbar_had_nominal.json
    nanoaods_ttbar_lep_nominal.json
    nanoaods_ttbar_semilep_nominal.json
    nanoaods_wjets_nominal.json

totals from nanoaods.json:
  signal__nominal  files=  2  nevts_total=    230000  nevts_wt_total=     230000.00
  ttbar_semilep__nominal  files=  2  nevts_total=   2598000  nevts_wt_total=  781902176.00
  ttbar_had__nominal  files=  2  nevts_total=   2430000  nevts_wt_total=  762688864.00
  ttbar_lep__nominal  files=  2  nevts_total=   1396000  nevts_wt_total=  100679184.00
  wjets__nominal  files=  2  nevts_total=   2140682  nevts_wt_total=256642023424.00
  data__nominal  files=  2  nevts_total=   4030719  nevts_wt_total=    4030719.00
